# Chapter Exercises

Write the following functions. You’ll want to use your own `State` type for which you’ve defined the Functor, Applicative, and Monad.

In [3]:
newtype MyState s a = MyState { runMyState :: s -> (a, s) }

instance Functor (MyState s) where
    fmap :: (a -> b) -> MyState s a -> MyState s b
    fmap f myState = MyState g
        where
            g s = (f a, newS)
                where
                    (a, newS) = runMyState myState s

instance Applicative (MyState s) where
    pure :: a -> MyState s a
    pure x = MyState (x,)

    (<*>) :: MyState s (a -> b) -> MyState s a -> MyState s b
    (<*>)  (MyState f) (MyState g) = MyState h
        where
            h s = (b, s_2)
                where
                    (f_ab, s_1) = f s
                    (a, s_2)    = g s_1
                    b           = f_ab a


instance Monad (MyState s) where
    return = pure

    (>>=) :: MyState s a -> (a -> MyState s b) -> MyState s b
    (>>=) (MyState f) f_aSb = MyState g
        where
            g s = (b, s_2)
                where
                    (a, s_1) = f s
                    stateSB  = f_aSb a
                    (b, s_2) = runMyState stateSB s_1

1. Construct a State where the state is also the value you return.

```haskell
get :: State s s
get = ???
```

Expected output:

```
Prelude> runState get "curryIsAmaze"
("curryIsAmaze","curryIsAmaze")
```

In [7]:
get :: MyState s s
get = MyState (\s -> (s,s))

In [10]:
runMyState get "curryIsAmaze"

("curryIsAmaze","curryIsAmaze")

2. Construct a State where the resulting state is the argument provided and the value is defaulted to unit.

```haskell
put :: s -> State s ()
put s = ???
```
```
Prelude> runState (put "blah") "woot"
((),"blah")
```

In [19]:
put :: s -> MyState s ()
put s = MyState (const ((), s))

In [21]:
runMyState (put "blah") "woot"

((),"blah")

3. Run the State with 𝑠 and get the state that results.

```haskell
exec :: State s a -> s -> s
exec (State sa) s = ???
```

```
Prelude> exec (put "wilma") "daphne"
"wilma"
Prelude> exec get "scooby papu"
"scooby papu"
```

In [22]:
exec :: MyState s a -> s -> s
exec (MyState sa) = snd . sa

In [23]:
exec (put "wilma") "daphne"

"wilma"

In [24]:
exec get "scooby papu"

"scooby papu"

4. Run the State with 𝑠 and get the value that results.

```haskell
eval :: State s a -> s -> a
eval (State sa) = ???
```

```
Prelude> eval get "bunnicula"
"bunnicula"
Prelude> eval get "stake a bunny"
"stake a bunny"
```

In [25]:
eval :: MyState s a -> s -> a
eval (MyState sa) = fst . sa

In [26]:
eval get "bunnicula"

"bunnicula"

In [27]:
eval get "stake a bunny"

"stake a bunny"

5. Write a function which applies a function to create a new State.

```haskell
modify :: (s -> s) -> State s ()
modify = undefined
```

Should behave like the following:
```
Prelude> runState (modify (+1)) 0
((),1)
Prelude> runState (modify (+1) >> modify (+1)) 0
((),2)
```
You don’t need to compose them, you can throw away the result
because it returns unit for 𝑎 anyway.

In [6]:
modify :: (s -> s) -> MyState s ()
modify f_ss = MyState (\s -> ((), f_ss s))

In [7]:
runMyState (modify (+1)) 0

((),1)

In [8]:
runMyState (modify (+1) >> modify (+1)) 0

((),2)